# 33 — Calibración, ranking y congelamiento de selección

Divide temporalmente VALIDATION en dos partes: una para ajustar calibración y otra posterior para comparar modelos/calibradores. Al final congela `selected_model.json` **antes de cualquier acceso a TEST**.

In [1]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m3_gate = paths.docs / "decisions" / "M3_GATE.md"
assert m3_gate.exists() and "M3 ✅ CLOSED" in m3_gate.read_text(encoding="utf-8"), (
    "M3 no está formalmente cerrado. No iniciar Modeling."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
MODEL_DIR = paths.root / "artifacts" / "models" / "m4"
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

{
    "gold_schema": GOLD_SPEC.schema_version,
    "modeling_schema": MODEL_SPEC.schema_version,
    "target": MODEL_SPEC.target,
    "model_matrix": MODEL_SPEC.model_matrix,
    "operational_review_fraction": MODEL_SPEC.ranking.operational_fraction,
}

assert (paths.reports / "modeling" / "32_validation.parquet").exists(), "Ejecuta 32 primero."
from argos.models.calibration import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


'm4-33-calibration-ranking-v1'

In [2]:
from argos.models.calibration import build_calibration_and_selection

result = build_calibration_and_selection(
    reports_root=paths.reports,
    model_dir=MODEL_DIR,
    manifest_dir=MANIFEST_DIR,
    modeling_spec=MODEL_SPEC,
)
result.candidate_metrics


,model,split,rows,positives,prevalence,pr_auc,roc_auc,log_loss,brier,ece
0,logistic_sgd,validation_selection_raw,428076,1568,0.003663,0.016840,0.749332,1.129779,0.076759,0.077642
1,hist_gradient_boosting,validation_selection_raw,428076,1568,0.003663,0.035308,0.874320,0.112221,0.027281,0.077609


In [3]:
result.calibration_metrics

,model,split,rows,positives,prevalence,pr_auc,roc_auc,log_loss,brier,ece,calibration_method
0,hist_gradient_boosting+none,validation_selection,428076,1568,0.003663,0.035308,0.874320,0.112221,0.027281,0.077609,none
1,hist_gradient_boosting+sigmoid,validation_selection,428076,1568,0.003663,0.035308,0.874320,0.020866,0.003645,0.002428,sigmoid
2,hist_gradient_boosting+isotonic,validation_selection,428076,1568,0.003663,0.035580,0.864507,0.020681,0.003606,0.002340,isotonic


In [4]:
result.ranking

,model,split,review_fraction,reviewed,captured_positives,precision_at_k,recall_at_k,lift_at_k,cumulative_gain
0,hist_gradient_boosting+isotonic,validation_selection,0.001,429,25,0.058275,0.015944,15.909537,0.015944
1,hist_gradient_boosting+isotonic,validation_selection,0.005,2141,111,0.051845,0.070791,14.154063,0.070791
2,hist_gradient_boosting+isotonic,validation_selection,0.010,4281,267,0.062369,0.170281,17.027107,0.170281
3,hist_gradient_boosting+isotonic,validation_selection,0.025,10702,567,0.052981,0.361607,14.464151,0.361607
4,hist_gradient_boosting+isotonic,validation_selection,0.050,21404,797,0.037236,0.508291,10.165721,0.508291
5,hist_gradient_boosting+isotonic,validation_selection,0.100,42808,1053,0.024598,0.671556,6.715498,0.671556


In [5]:
result.reliability

,model,split,bin,rows,mean_predicted_probability,observed_rate,min_score,max_score
0,hist_gradient_boosting+isotonic,validation_selection,0,236132,0.002681,0.000597,1.000000e-07,0.002712
1,hist_gradient_boosting+isotonic,validation_selection,1,26262,0.002738,0.000647,2.738059e-03,0.002738
2,hist_gradient_boosting+isotonic,validation_selection,2,26751,0.002831,0.001346,2.789083e-03,0.002831
3,hist_gradient_boosting+isotonic,validation_selection,3,28408,0.002936,0.001936,2.859016e-03,0.002936
4,hist_gradient_boosting+isotonic,validation_selection,4,25127,0.003728,0.002746,3.360215e-03,0.004545
5,hist_gradient_boosting+isotonic,validation_selection,5,32053,0.005115,0.003993,4.873274e-03,0.005447
6,hist_gradient_boosting+isotonic,validation_selection,6,33578,0.012008,0.010066,6.242197e-03,0.019697
7,hist_gradient_boosting+isotonic,validation_selection,7,19765,0.052855,0.039666,1.983340e-02,0.130938


In [6]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,calibration_partitions_temporal,validation,True,True,"{""fit_max"": ""2024-06-30 00:00:00"", ""selection_...",calibration fit and model-selection evaluation...
1,candidate_models_compared,selection,True,True,"{""models"": [""logistic_sgd"", ""hist_gradient_boo...",both linear and boosting candidates are compar...
2,calibration_methods_compared,calibration,True,True,"{""methods"": [""none"", ""sigmoid"", ""isotonic""]}",all predeclared calibration methods are evaluated
3,selection_beats_prevalence_pr_auc,usefulness,True,True,"{""prevalence"": 0.0036629009801997774, ""require...",selected candidate PR-AUC exceeds the frozen p...
4,selection_lift_operational_k,usefulness,True,True,"{""lift"": 17.02710660297756, ""minimum"": 1.0, ""r...",selected candidate provides lift above random ...
5,selection_frozen_before_test,test_isolation,True,True,"{""selection_path"": ""/Users/macm1pro/Desktop/pr...",model and calibration choice are frozen before...


In [7]:
blocking = result.validation.loc[result.validation["blocking"]]
print("33 ✅ PASS" if result.ready_for_next_notebook else "33 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Frozen selection:", result.selection_path)
assert result.ready_for_next_notebook


33 ✅ PASS
Blocking checks: 6/6
Frozen selection: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/models/m4/selected_model.json
